In [2]:
import google.auth
import numpy as np
import pandas as pd
import pygris 
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [3]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [4]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [5]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

In [6]:


@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [7]:
@cache
def gcs_pandas():
    return GCSPandas()

In [8]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Load Data
### Census Block
STATEFP20

2‑digit FIPS code for the state
Example: 06 = California

COUNTYFP20

3‑digit FIPS code for the county
Example: 001 = Alameda County

TRACTCE20

Census tract code (6 digits, no decimal)

BLOCKCE20

Census block code (4 digits)

GEOID20

Full 15‑digit concatenated identifier:
STATEFP + COUNTYFP + TRACTCE + BLOCKCE

Example: 060014001001234

NAME20

Block name (usually same as block number)

MTFCC20

Feature class code indicating type of Census block
For blocks this is usually:

G5030 = Census Block



UR20

Urban/rural classification flag

U = Urban
R = Rural



UACE20

Urban Area Census Code
If the block is inside an urban area, this is the UA code
Example: 06004 for Oakland UA

UATYPE20

Urban Area Type

U = Urbanized area (50,000+ people)
C = Urban cluster (2,500–49,999 people)



FUNCSTAT20

Functional status of the block
For blocks:

S = Statistical entity (standard for census blocks)



ALAND20

Land area in square meters

AWATER20

Water area in square meters

INTPTLAT20

Internal point latitude (label point for the block)

INTPTLON20

Internal point longitude

HOUSING20

Number of housing units in the block (2020 Census)

In [ ]:
def chunked(seq, size):
    """Yield successive chunks of length 'size' from a sequence."""
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [ ]:
def load_ca_blocks(year:int, variables:str, county_code: str):
    """
    Load all census blocks for California (2020 PL94 decennial).
    Downloads one county at a time because the API does not allow
    state-level block requests.
    """
    df = pygris.blocks(
        state="06",     # California
        county=county_code,   
        year=year,
        cache=True
       )

    # Reproject
    df = df.to_crs(geography_utils.CA_NAD83Albers_ft)

    # Buffer
    df["b250"] = df.buffer(250)
    df = df.drop(columns = ["geometry"])

    # Save locally first
    df.to_parquet("./chunked.parquet")

    # Save to GCS
    ca_counties = to_snakecase(pygris.counties(state="CA", year=year)[["NAME","COUNTYFP"]])
    file_name = ca_counties.loc[ca_counties.countyfp == county_code].name.iloc[0]
    df["county_name"] = file_name
    
    # Put the local file into the GCS bucket
    df.to_parquet("./chunked.parquet")
    fs.put("./chunked.parquet", f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_county_{file_name}_{year}.parquet")
    return df

In [ ]:
def _parse_gcs_path(gcs_path: str):
    """
    Split a GCS URL into (bucket, prefix) without leading 'gs://'.
    """
    if not gcs_path.startswith("gs://"):
        raise ValueError(f"Expected a 'gs://' path, got: {gcs_path}")
    no_scheme = gcs_path[5:]
    bucket, *rest = no_scheme.split("/", 1)
    prefix = rest[0] if rest else ""
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    return bucket, prefix

In [ ]:
def list_gcs_files(gcs_folder: str, extensions: Optional[List[str]] = None) -> list:
    """
    List all files in a GCS 'folder' (prefix). Optionally filter by extensions.
    Returns full 'gs://...' URIs.
    """
    bucket_name, prefix = _parse_gcs_path(gcs_folder)
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    uris: List[str] = []
    for blob in client.list_blobs(bucket_name, prefix=prefix):
        # Skip "directory placeholders"
        name = blob.name
        if name.endswith("/"):
            continue
        if extensions:
            if not any(name.lower().endswith(ext.lower()) for ext in extensions):
                continue
        uris.append(f"gs://{bucket_name}/{name}")

    return sorted(uris)

In [ ]:

try:
    import geopandas as gpd
    _HAS_GPD = True
except Exception:
    _HAS_GPD = False


In [ ]:
def concat_gcs_folder(
    gcs_folder: str = "gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset: bool = True,
    file_types: Optional[List[str]] = None,
    geometry: bool = False,
    dtype_overrides: Optional[dict] = None,
    use_threads: bool = True,
) -> Union[pd.DataFrame, "gpd.GeoDataFrame"]:
    """
    Concatenate all files in a GCS folder into a single DataFrame.
    """

    # Default supported formats
    if file_types is None:
        file_types = ["parquet", "csv", "feather", "geojson", "json"]

    files = list_gcs_files(gcs_folder, extensions=[f".{ext}" for ext in file_types])

    if not files:
        raise FileNotFoundError(f"No files found under: {gcs_folder} with types {file_types}")

    # Disable arrow.dataset for parquet — because GCS credentials fail there
    all_parquet = all(f.lower().endswith(".parquet") for f in files)
    
    # Updated behavior: Always use gcs_geopandas().read_parquet for parquet files
    if all_parquet:
        frames = []
        for uri in files:
            frames.append(gcs_geopandas().read_parquet(uri))
        df = pd.concat(frames, ignore_index=True)

        if dtype_overrides:
            df = df.astype(dtype_overrides, errors="ignore")
        return df

    # Otherwise fall back to file-by-file reading
    frames: List[Union[pd.DataFrame, "gpd.GeoDataFrame"]] = []

    for uri in files:
        lower = uri.lower()

        if lower.endswith(".parquet"):
            frames.append(gcs_geopandas().read_parquet(uri))

        elif lower.endswith(".feather"):
            frames.append(pd.read_feather(uri))

        elif lower.endswith(".csv"):
            frames.append(pd.read_csv(uri, low_memory=False))

        elif lower.endswith(".geojson") or (lower.endswith(".json") and "geo" in os.path.basename(uri).lower()):
            if not _HAS_GPD:
                raise ImportError("geopandas not installed—install it or set geometry=False.")
            gdf = gpd.read_file(uri)
            frames.append(gdf)

        else:
            print(f"[concat_gcs_folder] Skipping unsupported file: {uri}")

    if not frames:
        raise FileNotFoundError(f"Found files, but none were readable with the allowed types: {file_types}")

    # If any frames are GeoDataFrames, concatenate as GeoDataFrame
    if _HAS_GPD and any(isinstance(f, gpd.GeoDataFrame) for f in frames):
        df = pd.concat(frames, ignore_index=True)
        if "geometry" in df.columns and not isinstance(df, gpd.GeoDataFrame):
            df = gpd.GeoDataFrame(df, geometry="geometry", crs=frames[0].crs if hasattr(frames[0], "crs") else None)
    else:
        df = pd.concat(frames, ignore_index=True)

    if dtype_overrides:
        df = df.astype(dtype_overrides, errors="ignore")

    df = to_snakecase(df)
    return df


In [ ]:
def load_census_blocks(year:int):
    """
    Load all of the census blocks in a loop for each county. 
    Save out the results to GCS and read them back in to combine
    into a single dataframe.
    """
    # Find county codes
    ca_counties = to_snakecase(pygris.counties(state="CA", year=2020)[["NAME","COUNTYFP"]])

    ca_counties_list = list(ca_counties.countyfp.unique())

    # Load the census blocks for each county in a loop and save it out.
    for county in ca_counties_list:
        block_gdf = load_ca_blocks(year, "P1_001N", county)
        print(f"Done with {county}")

    # Concatenate all of the counties together
    gdf = concat_gcs_folder(
    gcs_folder = "gs://calitp-analytics-data/data-analyses/equity_index/census_blocks",
    prefer_arrow_dataset =True,
    file_types = None,
    geometry = True,
    dtype_overrides = None,
    use_threads = True)

    gdf.to_parquet("./combined.parquet")
    fs.put("./combined.parquet", f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_combined_{year}.parquet")
    return gdf

In [ ]:
"""gdf = concat_gcs_folder(
    gcs_folder = "gs://calitp-analytics-data/data-analyses/equity_index/census_blocks",
    prefer_arrow_dataset =True,
    file_types = None,
    geometry = True,
    dtype_overrides = None,
    use_threads = True)

gdf.to_parquet("./combined.parquet")
fs.put("./combined.parquet", f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_combined_2020.parquet")"""

### Public Road Functional Classification
**Amanda** Need to fix: URL maxes out at 2000 rows when there are thousands more. 

In [ ]:
# https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson

In [ ]:
SHARED_DATA_GCS = "gs://calitp-analytics-data/data-analyses/shared_data/"

In [ ]:
def load_interstate_freeway():
    gdf = to_snakecase(gcs_geopandas().read_parquet(f"{SHARED_DATA_GCS}public_road_functional_classification.parquet")).to_crs(geography_utils.CA_NAD83Albers_ft)

    # Filte out what we don't need
    gdf2 = gdf.loc[gdf["f_system"].isin([1,2])]

    # Buffer
    gdf2["b50"] = gdf2.geometry.buffer(50).drop(columns = ["geometry"])
    gdf2 = gdf2.set_geometry("b50")
    return gdf2

In [ ]:
interstate_freeway = load_interstate_freeway()

### TIMS Data

In [ ]:
def load_tims_data():
    """
    Take all of the individual county CSV files and 
    input them into a single dataframe.
    """
    gdf = concat_gcs_folder(
    gcs_folder="gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset=True)

    # Turn into geodataframe
    gdf = gpd.GeoDataFrame(gdf, geometry=gpd.points_from_xy(gdf.point_x, gdf.point_y), crs=geography_utils.WGS84 ).to_crs(geography_utils.CA_NAD83Albers_ft)

    # Subset
    gdf = gdf[['case_id','collision_severity','geometry','accident_year']]
    # Save
    gdf.to_parquet("./combined.parquet")
    fs.put("./combined.parquet", "gs://calitp-analytics-data/data-analyses/equity_index/tims/tims_combined.parquet")
    return gdf

In [ ]:
# tims_data = load_tims_data()

## Overlay TIMS with Public Road Functional Classification data for crashes we don't want. 
* Filter them out

In [ ]:
def find_relevant_crashes(year:int):
    interstate_freeway_gdf = load_interstate_freeway()

    tims_gdf = to_snakecase(gcs_geopandas().read_parquet("gs://calitp-analytics-data/data-analyses/equity_index/tims/tims_combined.parquet"))

    # We only care about severe and fatal crashes
    tims_gdf2 = tims_gdf.loc[tims_gdf.collision_severity.isin([1,2])]

    # Sjoin to find the severe and fatal crashes on roads we don't care about
    m1 = (
        tims_gdf2.sjoin(interstate_freeway_gdf.to_crs(tims_gdf2.crs), how="inner", predicate="intersects")
        .reset_index(drop=True)
        .drop(columns=["index_right"])
    )

    # Keep only the crashes we care about
    crashes_to_delete_list = list(m1.case_id.unique())

    relevant_crashes_gdf = tims_gdf.loc[~tims_gdf.case_id.isin(crashes_to_delete_list)].reset_index()

    # Save
    relevant_crashes_gdf.to_parquet(f"./tims_relevant_crashes_{year}.parquet")
    fs.put("./tims_relevant_crashes_2026.parquet", f"gs://calitp-analytics-data/data-analyses/equity_index/analysis_{year}/tims_relevant_crashes_{year}.parquet")
    return relevant_crashes_gdf

In [ ]:
relevant_crashes_gdf = find_relevant_crashes(year=2026)

In [20]:
census_gdf = to_snakecase(gcs_geopandas().read_parquet(f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_combined_2020.parquet"))

In [24]:
len("060855117072009")

15

# Overlay crashes we are interested in with Census Block

In [9]:
def crashes_census_block(analysis_year:str, census_year:str):
    """
    Merge the crashes we are interested in with census blocks.

    Analysis_year = the year we are running this workflow
    Census_year = the year of Census Bureau data we are using.
    """
    crashes_gdf = to_snakecase(gcs_geopandas().read_parquet(f"gs://calitp-analytics-data/data-analyses/equity_index/analysis_{analysis_year}/tims_relevant_crashes_{analysis_year}.parquet"))

    census_gdf = to_snakecase(gcs_geopandas().read_parquet(f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_combined_{census_year}.parquet"))

    # Merge
    m1 = (
        crashes_gdf.sjoin(census_gdf, how="left", predicate="within")
        .reset_index(drop=True)
        .drop(columns=["index_right"])
    )

    # Groupby
    agg1 = m1.groupby(["geoid20"]).agg({"aland20":"max","case_id":"nunique"}).reset_index().rename(columns = {"case_id":"n_crashes"})

    # Find crash density
    agg1["crash_density"] = agg1["n_crashes"] / agg1["aland20"]

    #Rank
    agg1["crash_percentile_100"] = (
    agg1["crash_density"].rank(pct=True) * 100
    )

    return agg1

In [34]:
ranked_crashes = crashes_census_block(analysis_year = 2026, census_year = 2020)

In [11]:
ranked_crashes.shape

(285267, 5)

In [52]:
ranked_crashes['tractid'] = ranked_crashes['geoid20'].astype(str).str[:11]

# Load AADT Data

In [68]:
aadt_df = to_snakecase(gcs_geopandas().read_file("gs://calitp-analytics-data/data-analyses/equity_index/aadt/Annual_Average_Daily_Traffic.geojson")).to_crs(geography_utils.CA_NAD83Albers_ft)

In [70]:
aadt_df.columns

Index(['objectid', 'district', 'rte', 'rte_sfx', 'cnty', 'pm_pfx', 'pm',
       'pm_sfx', 'description', 'back_peak_hour', 'back_peak_madt',
       'back_aadt', 'ahead_peak_hour', 'ahead_peak_madt', 'ahead_aadt',
       'geometry'],
      dtype='object')

In [69]:
aadt_df.head(2)

,objectid,district,rte,rte_sfx,cnty,pm_pfx,pm,pm_sfx,description,back_peak_hour,back_peak_madt,back_aadt,ahead_peak_hour,ahead_peak_madt,ahead_aadt,geometry
0,1,5,1,,SB,R,0.001,,"LAS CRUCES, JCT. RTE. 101; MOBIL OIL PIER",40,600,540,860,8000,7700,POINT (-68029.904 7844978.379)
1,2,5,1,,SB,R,0.001,,"LAS CRUCES, JCT. RTE. 101; MOBIL OIL PIER",40,600,540,860,8000,7700,POINT (-68029.904 7844978.379)


# Join Crashes with Demographic Overlay
* Your Census 2020 GEOID20 looks like:
061050001021135  ← 15 digits (this is a block‑level GEOID)
* Your ACS 2023 GEOID looks like:
6019004210  ← 10 digits (this is a tract‑level GEOID)
* These two GEOIDs represent different levels of geography
* A block is nested inside a tract, but the IDs do not match.

In [14]:
# gs://calitp-analytics-data/data-analyses/equity_index/low_income_blockgroups_updated_acs2023.csv

In [15]:
demographic_overlay_df = to_snakecase(pd.read_csv("gs://calitp-analytics-data/data-analyses/equity_index/low_income_blockgroups_updated_acs2023.csv"))

In [16]:
demographic_overlay_df.geoid = demographic_overlay_df.geoid.astype(str)

In [50]:
demographic_overlay_df['tractid2'] = demographic_overlay_df['tractid'].astype(str).str.zfill(11)

In [51]:
demographic_overlay_df[["geoid", "tractid", "tractid2"]].sample(5)

,geoid,tractid,tractid2
9750,6037620904,6037620904,06037620904
21032,6079012302,6079012302,06079012302
7114,6037403722,6037403722,06037403722
867,6001441521,6001441521,06001441521
25080,6111003616,6111003616,06111003616


In [53]:
ranked_crashes[["tractid"]].sample(3)

,tractid
154249,06065041003
110712,06037901010
237356,06083003005


In [57]:
ranked_crashes.shape

(285267, 7)

In [58]:
demographic_overlay_df.shape

(25607, 18)

In [59]:
demographic_overlay_df.head()

,unnamed:_0,geoid,median_hh_income_bg,rounded_hh_size_bg,localized_income_screen_bg,state_income_screen_bg,income_screen_bg,tractid,median_hh_income_tract,rounded_hh_size_tract,localized_income_screen_tract,state_income_screen_tract,income_screen_tract,income_screen_final,income_screen_geo,geoid_v2,geoid2,tractid2
0,0,6001400100,250001.00,3,0.00,0,0,6001400100,250001.00,2,0.00,0,0,0,block group,06001400100,006001400100,06001400100
1,1,6001400100,242813.00,2,0.00,0,0,6001400100,250001.00,2,0.00,0,0,0,block group,06001400100,006001400100,06001400100
2,2,6001400200,226094.00,2,0.00,0,0,6001400200,225880.00,2,0.00,0,0,0,block group,06001400200,006001400200,06001400200
3,3,6001400200,225789.00,2,0.00,0,0,6001400200,225880.00,2,0.00,0,0,0,block group,06001400200,006001400200,06001400200
4,4,6001400300,250001.00,3,0.00,0,0,6001400300,157731.00,2,0.00,0,0,0,block group,06001400300,006001400300,06001400300


In [62]:
demographic_overlay_df.shape, demographic_overlay_df.tractid.nunique()

((25607, 18), 9129)

In [60]:
ranked_crashes.head()

,geoid20,aland20,n_crashes,crash_density,crash_percentile_100,geoid_bg,tractid
0,060014001001001,695414.00,2,0.00,6.85,060014001001,06001400100
1,060014001001002,9330.00,1,0.00,49.87,060014001001,06001400100
2,060014001001003,1232350.00,1,0.00,3.01,060014001001,06001400100
3,060014001001010,1121172.00,7,0.00,10.43,060014001001,06001400100
4,060014001001011,772665.00,4,0.00,9.44,060014001001,06001400100


In [61]:
ranked_crashes.shape, ranked_crashes.geoid20.nunique()

((285267, 7), 285267)

* Original 15 digit `geoid20` is
Break it down using Census rules (blocks = 15-digit GEOID):

06 = state
105 = county
000102 = tract
1 = block group
135 = block

The tract portion = first 11 digits:

In [54]:
m1 = pd.merge(ranked_crashes, demographic_overlay_df, left_on = "tractid", right_on = "tractid2", how = "left", indicator = True)

In [55]:
m1._merge.value_counts()

_merge
both          874106
left_only          0
right_only         0
Name: count, dtype: int64